In [1]:
from langchain_nvidia_ai_endpoints.chat_models import ChatNVIDIA
from langchain.output_parsers import PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import List

c:\Users\likgn\anaconda3\envs\agent_for_teacher\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
class MutipleChoiceQuestion(BaseModel):
    question: str = Field(default="", description="The title of the question")
    options: List[str] = Field(description="4 options for multiple choice question")

json_schema = {
        "title": "joke",
        "description": "Joke to tell user.",
        "type": "object",
        "properties": {
            "setup": {
                "type": "string",
                "description": "The setup of the joke",
            },
            "punchline": {
                "type": "string",
                "description": "The punchline to the joke",
            },
        },
        "required": ["setup", "punchline"],
    }

pydantic_output_parser = PydanticOutputParser(pydantic_object=MutipleChoiceQuestion)

LLM output = {
    quesstion: "1 + 1 = ?",
    options: [2, 3, 4, 5, 6]
},
{
    quesstion: "1 + 3 = ?",
    options: [2, 3, 4, 5, 6]
}

LLM output -> Parser -> bien 

In [6]:
print(pydantic_output_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"question": {"default": "", "description": "The title of the question", "title": "Question", "type": "string"}, "options": {"description": "4 options for multiple choice question", "items": {"type": "string"}, "title": "Options", "type": "array"}}, "required": ["options"]}
```


# NVIDIA AI ENDPOINT GPTOSS

In [4]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

gpt_20b_nvidia = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key="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", 
  temperature=1,
  top_p=1,
  max_completion_tokens=4096,
)
gpt_20b_nvidia_structured = gpt_20b_nvidia.with_structured_output(schema=json_schema)

In [28]:
response = gpt_20b_nvidia_structured.invoke("Tell me a joke please.")
response

OutputParserException: Invalid json output: Sure thing! 😄

Why did the scarecrow win an award?  

Because he was outstanding in his field!… and he always gives the “corn‑y”est jokes.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

-> Đầu ra của mô hình chỉ đơn thuần là một chuỗi nên có vẻ Nvidia chưa cập nhật hỗ trợ cho mô hình này

- Sử dụng Prompt để mô tả structure output dựa trên quy định OpenAI Harmorny Response Format

In [62]:
messages = [
    SystemMessage(f"Reasoning: Medium. <|start|>developer<|message|># Response format \\n {pydantic_output_parser.get_format_instructions()}. Wrap your response in ```json```<|end|>"),
    HumanMessage("Generate a MCQ with 10 options about the mathematic formula behind Batch Normalize in LLM for a PhD. Using Latex format.")
]
response = gpt_20b_nvidia.invoke(messages)
print(response)

content='```json\n{\n  "question": "What is the correct mathematical formula behind Batch Normalization in large language models?",\n  "options": [\n    "$\\\\hat{x} = \\\\frac{x - \\\\mu_{\\\\text{batch}}}{\\\\sqrt{\\\\sigma_{\\\\text{batch}}^2 + \\\\varepsilon}},\\\\quad y = \\\\gamma \\\\hat{x} + \\\\beta$",\n    "$\\\\hat{x} = \\\\frac{x - \\\\mu_{\\\\text{batch}}}{\\\\sqrt{\\\\sigma_{\\\\text{batch}}^2 + \\\\varepsilon}},\\\\quad y = \\\\hat{x} + \\\\beta$",\n    "$\\\\hat{x} = \\\\frac{x - \\\\mu_{\\\\text{epoch}}}{\\\\sqrt{\\\\sigma_{\\\\text{epoch}}^2 + \\\\varepsilon}},\\\\quad y = \\\\gamma \\\\hat{x} + \\\\beta$",\n    "$\\\\hat{x} = \\\\frac{x - \\\\mu_{\\\\text{batch}}}{\\\\sigma_{\\\\text{batch}} + \\\\varepsilon},\\\\quad y = \\\\gamma \\\\hat{x} + \\\\beta$",\n    "$\\\\hat{x} = \\\\frac{x - \\\\mu_{\\\\text{batch}}}{\\\\sqrt{\\\\sigma_{\\\\text{batch}}^2 + \\\\varepsilon}},\\\\quad y = \\\\gamma x + \\\\beta$",\n    "$\\\\hat{x} = \\\\frac{x - \\\\mu_{\\\\text{batch}}}

In [63]:
pydantic_output_parser.parse(response.content)

MutipleChoiceQuestion(question='What is the correct mathematical formula behind Batch Normalization in large language models?', options=['$\\hat{x} = \\frac{x - \\mu_{\\text{batch}}}{\\sqrt{\\sigma_{\\text{batch}}^2 + \\varepsilon}},\\quad y = \\gamma \\hat{x} + \\beta$', '$\\hat{x} = \\frac{x - \\mu_{\\text{batch}}}{\\sqrt{\\sigma_{\\text{batch}}^2 + \\varepsilon}},\\quad y = \\hat{x} + \\beta$', '$\\hat{x} = \\frac{x - \\mu_{\\text{epoch}}}{\\sqrt{\\sigma_{\\text{epoch}}^2 + \\varepsilon}},\\quad y = \\gamma \\hat{x} + \\beta$', '$\\hat{x} = \\frac{x - \\mu_{\\text{batch}}}{\\sigma_{\\text{batch}} + \\varepsilon},\\quad y = \\gamma \\hat{x} + \\beta$', '$\\hat{x} = \\frac{x - \\mu_{\\text{batch}}}{\\sqrt{\\sigma_{\\text{batch}}^2 + \\varepsilon}},\\quad y = \\gamma x + \\beta$', '$\\hat{x} = \\frac{x - \\mu_{\\text{batch}}}{\\sqrt{\\sigma_{\\text{batch}}^2 + \\varepsilon}},\\quad y = \\gamma x - \\beta$', '$\\hat{x} = \\frac{x - \\mu_{\\text{batch}}}{\\sqrt{\\sigma_{\\text{batch}}^2 

In [66]:
messages = [
    SystemMessage(f"Reasoning: Medium. # Response format \\n {pydantic_output_parser.get_format_instructions()}. Wrap your response in ```json```"),
    HumanMessage("Generate a MCQ with 4 options about the mathematic formula behind Batch Normalize in LLM for a PhD. Using Latex format.")
]
response = gpt_20b_nvidia.invoke(messages)
print(response)

content='```json\n{\n  "question": "Which of the following equations correctly represents the Batch Normalization formula applied to the pre‑activation $z$ of a transformer layer in large language models during training?",\n  "options": [\n    "\\\\hat{z}_i = \\\\frac{z_i - \\\\mu_B}{\\\\sqrt{\\\\sigma_B^2 + \\\\epsilon}}\\\\;\\\\gamma + \\\\beta",\n    "\\\\hat{z}_i = \\\\frac{z_i - \\\\mu_B}{\\\\sqrt{\\\\sigma_B + \\\\epsilon}}\\\\;\\\\gamma + \\\\beta",\n    "\\\\hat{z}_i = \\\\frac{z_i - \\\\mu}{\\\\sqrt{\\\\sigma^2 + \\\\epsilon}}\\\\;\\\\gamma + \\\\beta",\n    "\\\\hat{z}_i = \\\\frac{z_i - \\\\mu_B}{\\\\sqrt{\\\\sigma_B^2 + \\\\epsilon}}\\\\;\\\\beta + \\\\gamma"\n  ]\n}\n```' additional_kwargs={'reasoning_content': 'We must output JSON with properties: "question" (string) and "options" (array of strings). Required is "options", but "question" is default empty? The schema sets default "", description ends. It is not required. But must be present? It says required: ["options"]. 

In [4]:
pydantic_output_parser.parse(response.content)

NameError: name 'response' is not defined

## Sử dụng raw message thay vì phải sử dụng thông qua langchain

In [69]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "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"
)

completion = client.chat.completions.create(
  model="openai/gpt-oss-20b",
  messages=[{"role":"developer","content":"Reasoning: Medium. # Response format \\n {pydantic_output_parser.get_format_instructions()}. Wrap your response in ```json```"},
            {"role":"system", "content": "Generate a MCQ with 4 options about the mathematic formula behind Batch Normalize in LLM for a PhD. Using Latex for Markdownformat."}],
  temperature=1,
  top_p=1,
  max_tokens=4096,
  stream=True
)

for chunk in completion:
  reasoning = getattr(chunk.choices[0].delta, "reasoning_content", None)
  if reasoning:
    print(reasoning, end="")
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")



We need to generate a multiple choice question (MCQ) with 4 options about the mathematical formula behind Batch Normalization in LLM (Large Language Models). It should be suitable for a PhD-level audience. Must use LaTeX for Markdown format.

We need to produce a question about the mathematical formula behind Batch Normalization. Provide 4 options, presumably with the correct answer and maybe distractors that reflect common misunderstandings or variations. It should involve the formula for BN used in transformer-based LLM. That formula includes the scaling and shifting parameters after normalization: $\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$ followed by $\gamma \hat{x}_i + \beta$. For LLM, the batch dimension is over tokens across batch and sequence perhaps. Also the formula for computing the mean and variance across input batch (e.g., across tokens). Could include per-feature channel.

Also the typical formula is:

$\hat{x}_i = \frac{x_i - \mu_{\mathcal{B}}}{\sqrt

# Một model khác trên Nvidia

In [ ]:
# List models in the NVIDIA AI Endpoints
ChatNVIDIA.get_available_models()

In [66]:
for model in ChatNVIDIA.get_available_models():
    if model.supports_structured_output:
        print(model)

id='mistralai/mistral-large-2-instruct' model_type='chat' client='ChatNVIDIA' endpoint=None aliases=None supports_tools=True supports_structured_output=True supports_thinking=False base_model=None
id='openai/gpt-oss-20b' model_type='chat' client='ChatNVIDIA' endpoint=None aliases=None supports_tools=True supports_structured_output=True supports_thinking=False base_model=None
id='openai/gpt-oss-120b' model_type='chat' client='ChatNVIDIA' endpoint=None aliases=None supports_tools=True supports_structured_output=True supports_thinking=False base_model=None
id='institute-of-science-tokyo/llama-3.1-swallow-70b-instruct-v0.1' model_type='chat' client='ChatNVIDIA' endpoint=None aliases=None supports_tools=False supports_structured_output=True supports_thinking=False base_model=None
id='nv-mistralai/mistral-nemo-12b-instruct' model_type='chat' client='ChatNVIDIA' endpoint=None aliases=None supports_tools=True supports_structured_output=True supports_thinking=False base_model=None
id='nvidia/ll

In [8]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llama_8b_invidia = ChatNVIDIA(
  model="meta/llama-3.1-70b-instruct",
  api_key="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", 
  temperature=1,
  top_p=1,
  max_completion_tokens=100000,
)
llama_8b_invidia_structured = llama_8b_invidia.with_structured_output(schema=json_schema)

In [9]:
response = llama_8b_invidia_structured.invoke("Tell me a joke about Tarot")
response

{'setup': 'Why did the tarot reader break up with her boyfriend?',
 'punchline': 'Because their relationship was always drawing the same negative cards!'}

In [22]:
llama_8b_invidia_structured = llama_8b_invidia.with_structured_output(schema=MutipleChoiceQuestion)
response = llama_8b_invidia_structured.invoke("Generate a MCQ with 4 options about the mathematic formula behind Convolution in LLM for a PhD. Using Latex format")

In [23]:
response

MutipleChoiceQuestion(question='What is the mathematical formula behind convolution in Large Language Models (LLMs)?', options=['(a) $\\mathbf{h}_t = \\mathbf{W} \\\\[ \\mathbf{x}_t \\circledast \\boldsymbol{\\phi} ( \\mathbf{x}_{t-1}) \\\\', '(b) $\\mathbf{h}_t = \\mathbf{W} \\otimes [ \\mathbf{x}_t \\oplus \\boldsymbol{\\phi} ( \\mathbf{x}_{t-1}) ]$', '(c) $\\mathbf{h}_t = \\mathbf{W} \\[ \\mathbf{x}_t \\ast \\boldsymbol{\\phi} ( \\mathbf{x}_{t-1}) ]$', '(d) $\\mathbf{h}_t = \\mathbf{W} \\cdot [ \\mathbf{x}_t \\odot \\boldsymbol{\\phi} ( \\mathbf{x}_{t-1}) ]$', 'answer Explanation: The correct answer is option (c). The mathematical formula behind convolution in LLMs involves a sliding window approach, where the input sequence is convolved with learned filters using the convolution operation $\\ast$. The output of this convolution is then transformed using the weights $\\mathbf{W}$ to produce the final output $\\mathbf{h}_t$ . ', 'formula notation guide: $\\mathbf{x}_t$ is the input s

- Kết quả khi sử dụng mô hình llama-3.1-8b-instruct khả quan tuy, nhiên khi sử dụng mô hình meta/llama-3.3-70b-instruct thì gặp lỗi tương tự GPT

In [24]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llama_8b_invidia = ChatNVIDIA(
  model="meta/llama-3.3-70b-instruct",
  api_key="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", 
  temperature=1,
  top_p=1,
  max_completion_tokens=100000,
)
llama_8b_invidia_structured = llama_8b_invidia.with_structured_output(schema=json_schema)

In [29]:
llama_8b_invidia_structured = llama_8b_invidia.with_structured_output(schema=MutipleChoiceQuestion)
response = llama_8b_invidia_structured.invoke("Generate a MCQ with 4 options about the mathematic formula behind Convolution in LLM for a PhD. Using Latex format")

In [30]:
response

# ChatGoogleGenerativeAI

In [29]:
MutipleChoiceQuestion.model_json_schema()

{'properties': {'question': {'default': '',
   'description': 'The title of the question',
   'title': 'Question',
   'type': 'string'},
  'options': {'description': '4 options for multiple choice question',
   'items': {'type': 'string'},
   'title': 'Options',
   'type': 'array'}},
 'required': ['options'],
 'title': 'MutipleChoiceQuestion',
 'type': 'object'}

In [37]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

In [38]:
structured_llm = llm.with_structured_output(MutipleChoiceQuestion)

In [45]:
response = structured_llm.invoke("Generate a MCQ about the mathematic formula behind Batch Normalize in LLM")

In [ ]:
str_response = response.question + "\\n" + "\\n".join(response.options)

In [52]:
print(str_response)

Which of the following formulas represents the normalization step in Batch Normalization, where $\mu_B$ is the mini-batch mean, $\sigma_B^2$ is the mini-batch variance, $x_i$ is an input activation, and $\epsilon$ is a small constant for numerical stability?\nA) $y_i = \gamma x_i + \beta$\nB) $\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$\nC) $\mu_B = \frac{1}{m}\sum_{i=1}^m x_i$\nD) $\sigma_B^2 = \frac{1}{m}\sum_{i=1}^m (x_i - \mu_B)^2$


# Sử dụng bind tool

Langchain khuyên rằng cách truyền Ouput Instruction vào Prompt đôi khi không phải là cách được khuyến khích. Do đó, hãy thử sử dụng tính năng bind_tool
https://github.com/langchain-ai/langchain/blob/v0.3/docs/docs/concepts/structured_outputs.mdx

In [2]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

In [18]:


gpt_20b_nvidia = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key="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", 
  temperature=1,
  top_p=1,
  max_completion_tokens=10000,
)
gpt_20b_nvidia_structured = gpt_20b_nvidia.bind_tools([MutipleChoiceQuestion])

In [31]:
response = gpt_20b_nvidia_structured.invoke("Generate a MCQ with 4 options about the mathematic formula behind Batch Normalize in LLM for a PhD. Using Latex format for math expression. Aways reponse using tool calling")

In [28]:
print(response)

content='' additional_kwargs={'reasoning_content': 'The user: "Generate a MCQ with 4 options about the mathematic formula behind Batch Normalize in LLM for a PhD. Using Latex format for math expression. Aways reponse using tool calling"\n\nWe have a tool: MutipleChoiceQuestion. We need to produce a JSON with question string and options array. Must use LaTeX for math.\n\nNote: The user says "Aways reponse using tool calling". That is, always respond with a tool call. They want us to call the MutipleChoiceQuestion function. The function expects a dict with "question" as optional, "options" as array of 4 strings. We need to generate LaTeX math expression within option text perhaps. The question may be about "the mathematically formula behind Batch Normalization in LLM".\n\nWe generate a question that asks what formula is used for batch normalization in lay terms: maybe question: "What is the correct expression for computing the batch-normalized activation in a transformer layer?" Option A

In [32]:
response.tool_calls[0]["args"]

IndexError: list index out of range

In [25]:
pydantic_output = MutipleChoiceQuestion.model_validate(response.tool_calls[0]["args"])

In [26]:
pydantic_output

MutipleChoiceQuestion(question='\\n$\\displaystyle \\text{Which of the following is the correct expression for Batch Normalization applied to a vector of activations }x\\text{ in a transformer layer (ignoring the affine parameters }\\gamma\\text{ and }\\beta\\text{)?}$', options=['$\\displaystyle \\frac{x - \\mu_B}{\\sqrt{\\sigma_B^2 + \\epsilon}}$', '$\\displaystyle \\frac{x - \\mu_B}{\\sigma_B + \\epsilon}$', '$\\displaystyle \\frac{\\mu_B - x}{\\sqrt{\\sigma_B^2 + \\epsilon}}$', '$\\displaystyle \\frac{x}{\\mu_B + \\sigma_B}$'])

# ChatNVIDIA method in structured output

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

gpt_20b_nvidia = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key="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", 
  temperature=1,
  top_p=1,
  max_completion_tokens=4096,
)
gpt_20b_nvidia_structured = gpt_20b_nvidia.with_structured_output()

# Test 